# 06 — Bitcoin, PowerGrid: intentando romper el efecto (falsación)

**Pregunta que se probó:** Si el efecto depende de la heterogeneidad de
grado, debería DEBILITARSE o DESAPARECER en redes casi regulares. Este
notebook prueba deliberadamente romper la hipótesis con PowerGrid (grado casi
uniforme) y contrasta con bitcoin (grado muy heterogéneo).

**Resultado:** SOBREVIVIÓ la falsación — en el buen sentido. En PowerGrid
(CV de grado bajo) los observables V y τ̃ se vuelven inconsistentes entre sí
(discrepan en signo), señal de que el efecto se debilita cuando no hay
heterogeneidad de grado que explotar. En bitcoin (CV alto) el efecto se
mantiene fuerte. Esto acota el fenómeno: sin heterogeneidad de grado, no hay
anti-centralidad clara.

Ver detalle completo en `paper/SPG_final_v9.docx`, Sección 5.3.


## Fuentes de datos

- **PowerGrid**
  https://networks.skewed.de/net/power
- **Bitcoin alpha**
  https://networks.skewed.de/net/bitcoin_alpha


> **Nota:** de cada dataset se usó únicamente el archivo de tipo `edges` (lista de aristas). No se usaron archivos de nodos ni de metadatos adicionales de Netzschleuder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/Romper'
print("¿existe la carpeta?:", os.path.isdir(CARPETA))
if os.path.isdir(CARPETA):
    print("archivos que veo:")
    for f in sorted(os.listdir(CARPETA)):
        print("  ", f)

In [ ]:
import pandas as pd, networkx as nx, numpy as np, os
from scipy.linalg import eigh
from scipy.stats import spearmanr

class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N; self.lambda2 = ev[k0]

def medir(archivo, nombre):
    df = pd.read_csv(archivo, comment='#', header=None)
    # tomar solo las 2 primeras columnas (source, target) sin importar cuántas haya
    df = df.iloc[:, :2]; df.columns = ['s','t']
    G = nx.Graph(); G.add_edges_from(df[['s','t']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
    A = nx.to_numpy_array(G); s = SPG(A)
    N = s.N; deg = s.degree
    rv  = spearmanr(deg, s.V)[0]
    rtt = spearmanr(deg, s.tau_tilde)[0]
    dens = A.sum()/(N*(N-1)); cv = deg.std()/deg.mean()
    fiable = "SI" if np.sign(rv)==np.sign(rtt) else "NO-descartar"
    print(f"{nombre:24s} {N:5d} {dens:6.3f} {cv:5.2f} {rv:+7.3f} {rtt:+7.3f} {fiable:>12s}")
    return dict(red=nombre, N=N, dens=dens, cv=cv, sp_V=rv, sp_tt=rtt, fiable=fiable)

print("cargado OK — ahora corre el bloque de medir")


cargado OK — ahora corre el bloque de medir


In [ ]:
CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/Romper'  # ajusta si tu ruta es otra
import os

# primero confirmamos QUÉ archivos ve y la ruta correcta
print("archivos .csv en la carpeta:")
for f in sorted(os.listdir(CARPETA)):
    if f.endswith('.csv'):
        print("  ", f)
print("="*60)

resultados = []
print(f"{'red':24s} {'N':>5s} {'dens':>6s} {'CV':>5s} {'Sp(V)':>7s} {'Sp(tt)':>7s} {'fiable':>12s}")
print("-"*72)
for f in sorted(os.listdir(CARPETA)):
    if not f.endswith('.csv'): continue
    try:
        resultados.append(medir(os.path.join(CARPETA, f), f[:-4]))
    except Exception as e:
        print(f"{f[:-4]:24s} ERROR: {str(e)[:40]}")


archivos .csv en la carpeta:
   PowerGrid.csv
   bitcoin_alpha.csv
   bitcoin_trust .csv
red                          N   dens    CV   Sp(V)  Sp(tt)       fiable
------------------------------------------------------------------------
PowerGrid                 4941  0.001  0.67  -0.565  +0.477 NO-descartar
bitcoin_alpha             3775  0.002  2.68  -0.962  -0.877           SI
bitcoin_trust             5875  0.001  3.15  -0.959  -0.884           SI


In [ ]:
import networkx as nx, numpy as np
from scipy.stats import spearmanr
import networkx.algorithms.community as nxcom

def prueba_tau_vs_V(G, nombre):
    s = SPG(nx.to_numpy_array(G))
    nodes = list(G.nodes()); idx = {n:i for i,n in enumerate(nodes)}
    V, tt = s.V, s.tau_tilde
    print(f"\n=== {nombre} (N={len(nodes)}) ===")

    # detectar comunidades (Louvain)
    try:
        comms = nxcom.louvain_communities(G, seed=42)
        comm_of = {n:i for i,c in enumerate(comms) for n in c}
    except Exception as e:
        print("  no se pudo comunidad:", e); return

    # TAREA 1: participation coefficient (rol de conector entre módulos)
    #   nodo conector = conecta a muchas comunidades distintas
    part = []
    for n in nodes:
        degs = {}
        for nb in G.neighbors(n):
            c = comm_of[nb]; degs[c] = degs.get(c,0)+1
        k = G.degree(n)
        P = 1 - sum((d/k)**2 for d in degs.values()) if k>0 else 0
        part.append(P)
    part = np.array(part)

    # TAREA 2: within-module degree (hub de módulo)
    wmd = []
    for n in nodes:
        c = comm_of[n]
        same = [m for m in comms[c]]
        sub = G.subgraph(same)
        wmd.append(sub.degree(n) if n in sub else 0)
    wmd = np.array(wmd, float)

    # TAREA 3: es nodo frontera (tiene vecinos en otra comunidad)?
    frontera = np.array([1 if any(comm_of[nb]!=comm_of[n] for nb in G.neighbors(n)) else 0
                         for n in nodes], float)

    for tarea, y in [('participation', part), ('within_mod_deg', wmd), ('frontera', frontera)]:
        rV  = abs(spearmanr(V, y)[0])
        rtt = abs(spearmanr(tt, y)[0])
        gana = "  <-- TAU GANA" if rtt - rV > 0.15 else ""
        print(f"  {tarea:16s}  |V|={rV:.3f}  |tau|={rtt:.3f}{gana}")

# corre en 3-4 redes CON estructura de comunidades clara y lambda_max/lambda2 BAJO
# (importante: solo redes donde tau_tilde es válido)
# ejemplos: jazz_collab, celegans_metabolic, una budapest _1m
prueba_tau_vs_V(G, "tu_red")


NameError: name 'G' is not defined